# **🏠 Predict House Prices**

---

# Regression Modelling

## Objectives

- Fit and evaluate regression models to predict house `SalePrice`.
- Identify a model that provides suitable predictive performance on unseen data.

## Inputs

- Cleaned house prices dataset.
- Feature engineering decisions identified in the previous notebook.

## Outputs

- Train set (features and target).
- Test set (features and target).
- ML pipeline to predict house `SalePrice`.
- Feature importance plot.

---

## ⚙️ 2. Set Up the Notebook

### Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

---

## 📥 3. Load Data

Load the cleaned house prices dataset produced during the data cleaning stage.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("outputs/datasets/cleaned/house_prices_cleaned.csv")

print(df.shape)
df.head(3)

---

## 🔧 3. Define the ML Pipeline

### Feature Engineering and Modelling Pipeline

A pipeline is created to apply the feature engineering steps identified in the
previous notebook.

Specifically, the pipeline will:

- Encode the categorical features (`OrdinalEncoder` and `OneHotEncoder`)
- Apply numerical transformations (`LogTransformer` and `PowerTransformer`)
- Scale the features (`StandardScaler`)
- Select features using `SelectFromModel`
- Fit a regression model

In [ ]:
from sklearn.pipeline import Pipeline

# Feature Engineering
from feature_engine.encoding import OrdinalEncoder, OneHotEncoder
from feature_engine import transformation as vt

# Feature Scaling
from sklearn.preprocessing import StandardScaler

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# ML Algorithms
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)
from xgboost import XGBRegressor


def PipelineOptimization(model):
    """
    Creates a regression pipeline using the feature engineering decisions
    identified in the Feature Engineering notebook.
    """
    pipeline_base = Pipeline([

        # CATEGORICAL ENCODING
        ("OrdinalCategoricalEncoder", OrdinalEncoder(
    encoding_method='arbitrary',
    variables=[
        'BsmtExposure',
        'GarageFinish',
        'KitchenQual'
    ],
    unseen='encode'
)),

        ("OneHotCategoricalEncoder", OneHotEncoder(
            variables=['BsmtFinType1'],
            drop_last=False
        )),

        # NUMERICAL TRANSFORMATIONS
        ("LogTransformer", vt.LogTransformer(
            variables=[
                '1stFlrSF',
                'GrLivArea',
                'LotArea',
                'LotFrontage'
            ]
        )),

        ("PowerTransformer", vt.PowerTransformer(
            variables=[
                'BsmtUnfSF',
                'TotalBsmtSF'
            ]
        )),

        # FEATURE SCALING
        ("feat_scaling", StandardScaler()),

        # FEATURE SELECTION
        ("feat_selection", SelectFromModel(model)),

        # MODEL
        ("model", model),

    ])

    return pipeline_base

### Hyperparameter Optimisation Class

The `HyperparameterOptimizationSearch` class is adapted from the Code Institute Predictive Analytics course material.

The class works as follows:

- `__init__` receives the models and their hyperparameters.
- `fit` tests each model and its hyperparameters using `GridSearchCV`.
- `score_summary` summarises the performance of each model.

This allows us to compare the models and identify the best-performing configuration.

In [ ]:
from sklearn.model_selection import GridSearchCV


class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = PipelineOptimization(self.models[key])
            params = self.params[key]

            gs = GridSearchCV(
                model,
                params,
                cv=cv,
                n_jobs=n_jobs,
                verbose=verbose,
                scoring=scoring
            )

            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):

        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []

        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []

            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)

            for p, s in zip(params, all_scores):
                rows.append(row(k, s, p))

        df = pd.concat(rows, axis=1).T.sort_values(
            [sort_by], ascending=False
        )

        columns = [
            'estimator',
            'min_score',
            'mean_score',
            'max_score',
            'std_score'
        ]

        columns = columns + [
            c for c in df.columns if c not in columns
        ]

        return df[columns], self.grid_searches

---

## ✂️ 4. Train and Test Split

The dataset is split into **80% training data and 20% test data**.

The training set is used to train the models, while the test set is used to evaluate how well the models perform on unseen data.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['SalePrice'], axis=1),
    df['SalePrice'],
    test_size=0.2,
    random_state=0,
)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

The dataset has been split into:

- **Training set:** 1,168 observations with 21 predictor features.
- **Test set:** 292 observations with 21 predictor features.

### Train Set Target Distribution

The distribution of the training target `SalePrice` is inspected before modelling.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

sns.histplot(y_train, kde=True)
plt.title("Train Set SalePrice Distribution")
plt.xlabel("SalePrice")
plt.show()

### Target Distribution Interpretation

- `SalePrice` is positively skewed, with most houses priced between roughly 100,000 and 250,000 and fewer houses at the higher end.

- No transformation will currently be applied to `SalePrice` before modelling.

---

## 🔎 5. Quick Model Search

The first step is to compare a range of regression models using their default
hyperparameters.

### Model Performance Requirement

We agreed with the client that the model should achieve an R² score of at least
**0.75** on both the train and test sets.

### Define Models

We now define the regression models to compare, using their default hyperparameters.

In [ ]:
models_quick_search = {
    "LinearRegression": LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=0),
    "RandomForestRegressor": RandomForestRegressor(random_state=0),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
    "AdaBoostRegressor": AdaBoostRegressor(random_state=0),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=0),
    "XGBRegressor": XGBRegressor(random_state=0),
}

params_quick_search = {
    "LinearRegression": {},
    "DecisionTreeRegressor": {},
    "RandomForestRegressor": {},
    "ExtraTreesRegressor": {},
    "AdaBoostRegressor": {},
    "GradientBoostingRegressor": {},
    "XGBRegressor": {},
}

### Model Performance Requirement

The project requirement is to achieve an **R² score of at least 0.75 on both
the training and test sets**.

We now run the quick search using the training data.

The models are compared using their default hyperparameters, with `R²` used to measure performance and 5-fold cross-validation (`cv=5`).

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_quick_search,
    params=params_quick_search
)

search.fit(
    X_train,
    y_train,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

### Quick Search Results

We summarise the cross-validation results to compare the performance of the
different regression models.

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(
    sort_by='mean_score'
)

grid_search_summary

---

## ⚙️ 6. Hyperparameter Optimisation

Following the quick model search, we now perform a more extensive search on the most suitable models.

The aim is to find the best hyperparameter configuration using `GridSearchCV`.

### Define Models and Hyperparameters

`ExtraTreesRegressor` and `RandomForestRegressor` achieved the strongest
performance during the quick search and are therefore selected for further
optimisation.

The hyperparameter settings are based on the recommendations provided in the Code Institute Predictive Analytics course material.

In [ ]:
models_search = {
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
    "RandomForestRegressor": RandomForestRegressor(random_state=0),
}

params_search = {
    "ExtraTreesRegressor": {
        "model__n_estimators": [100, 50, 150],
        "model__max_depth": [None, 3, 15],
        "model__min_samples_split": [2, 50],
        "model__min_samples_leaf": [1, 50],
    },

    "RandomForestRegressor": {
        "model__n_estimators": [100, 50, 140],
        "model__max_depth": [None, 4, 15],
        "model__min_samples_split": [2, 50],
        "model__min_samples_leaf": [1, 50],
        "model__max_leaf_nodes": [None, 50],
    },
}

### Run Extensive Search

We now run the hyperparameter search using `R²` as the scoring metric and 5-fold cross-validation.

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_search,
    params=params_search
)

search.fit(
    X_train,
    y_train,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

### Hyperparameter Search Results

We summarise the results to identify the best-performing model and hyperparameter configuration.

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(
    sort_by='mean_score'
)

grid_search_summary

### Select Best Model and Parameters

We select the model with the highest mean R² score and retrieve its best hyperparameter configuration.

In [ ]:
best_model = grid_search_summary.iloc[0, 0]
best_model

In [ ]:
best_parameters = grid_search_pipelines[best_model].best_params_
best_parameters

In [ ]:
pipeline_reg = grid_search_pipelines[best_model].best_estimator_
pipeline_reg

### Hyperparameter Optimisation Interpretation

`ExtraTreesRegressor` produced the best mean R² score. 

---

## 📊 7. Feature Importance

Next, we will identify the most important features using `feature_importances_`.

In [ ]:
# After feature engineering, the feature space may change.
# Get the column names after the feature engineering steps.

data_cleaning_feat_eng_steps = 4

columns_after_data_cleaning_feat_eng = (
    Pipeline(pipeline_reg.steps[:data_cleaning_feat_eng_steps])
    .transform(X_train)
    .columns
)

# Get the features retained by SelectFromModel
best_features = columns_after_data_cleaning_feat_eng[
    pipeline_reg['feat_selection'].get_support()
].to_list()


# Create DataFrame to display feature importance
df_feature_importance = (
    pd.DataFrame(data={
        'Feature': best_features,
        'Importance': pipeline_reg['model'].feature_importances_
    })
    .sort_values(by='Importance', ascending=False)
)


# Most important features statement and plot
print(
    f"* These are the {len(best_features)} most important features "
    f"in descending order. The model was trained on them: "
    f"\n{df_feature_importance['Feature'].to_list()}"
)

df_feature_importance.plot(
    kind='bar',
    x='Feature',
    y='Importance'
)

plt.show()

### Interpretation of Results

The feature-selection stage selected nine features for use by the final regression model based on the `SelectFromModel` selection criterion.

`OverallQual` was the most important feature by a considerable margin, followed by `GrLivArea`, `GarageArea`, `YearBuilt` and `TotalBsmtSF`.

---

## 📈 8. Model Evaluation

The model is evaluated using:

- R² Score
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)

This will be done using our adapted Code Institute function.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error 
import numpy as np


def regression_performance(X_train, y_train, X_test, y_test, pipeline):
    print("Model Evaluation \n")
    print("* Train Set")
    regression_evaluation(X_train, y_train, pipeline)
    print("* Test Set")
    regression_evaluation(X_test, y_test, pipeline)


def regression_evaluation(X, y, pipeline):
    prediction = pipeline.predict(X)

    print('R2 Score:', r2_score(y, prediction).round(3))  
    print('Mean Absolute Error:', mean_absolute_error(y, prediction).round(3))  
    print('Mean Squared Error:', mean_squared_error(y, prediction).round(3))  
    print('Root Mean Squared Error:',
          np.sqrt(mean_squared_error(y, prediction)).round(3))
    print("\n")


def regression_evaluation_plots(
    X_train, y_train, X_test, y_test, pipeline, alpha_scatter=0.5
):
    pred_train = pipeline.predict(X_train)
    pred_test = pipeline.predict(X_test)

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

    sns.scatterplot(
        x=y_train, y=pred_train,
        alpha=alpha_scatter, ax=axes[0]
    )
    sns.lineplot(x=y_train, y=y_train, color='red', ax=axes[0])
    axes[0].set_xlabel("Actual")
    axes[0].set_ylabel("Predictions")
    axes[0].set_title("Train Set")

    sns.scatterplot(
        x=y_test, y=pred_test,
        alpha=alpha_scatter, ax=axes[1]
    )
    sns.lineplot(x=y_test, y=y_test, color='red', ax=axes[1])
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predictions")
    axes[1].set_title("Test Set")

    plt.show()

### Train and Test Performance

We evaluate the performance of both the train and test set. This is done by comparing the actual sales price to the predicted sales price.

In [ ]:
regression_performance(
    X_train, y_train,
    X_test, y_test,
    pipeline_reg
)

regression_evaluation_plots(
    X_train, y_train,
    X_test, y_test,
    pipeline_reg,
    alpha_scatter=0.5
)

### Model Evaluation Interpretation

**R² Score**

The model achieved an R² score of **0.859** on the train set and **0.782** on the test set. We agreed with the client an R² score of at least **0.75** on both the train and test sets. Therefore, we have satisfied this criterion.

**Prediction Error**

On the train set, the model's predicted sale price differs from the actual sale price by approximately **17,970** on average. On the test set, this increases to approximately **20,613** on average. The RMSE is higher than the MAE on both sets, indicating that some properties have relatively large prediction errors.

**Prediction Plots**

From the graphs, we can see that the model performs well when predicting houses at the lower end of the price range. However, for some of the more expensive houses, the model struggles to predict the sale price accurately.

---

## 🔁 9. Refit with the Most Important Features

We now know which algorithm, hyperparameters and features best fit our data.

The model will now be refitted using only the most important features identified in the previous analysis.

### Create Train and Test Sets

We split the data again using the same 80/20 train and test split.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['SalePrice'], axis=1),
    df['SalePrice'],
    test_size=0.2,
    random_state=0
)

print(
    "* Train set:", X_train.shape, y_train.shape,
    "\n* Test set:", X_test.shape, y_test.shape
)

### Filter to Selected Features

Now we subset the best features.

In [ ]:
X_train = X_train.filter(best_features)
X_test = X_test.filter(best_features)

print(
    "* Train set:", X_train.shape, y_train.shape,
    "\n* Test set:", X_test.shape, y_test.shape
)

X_train.head(3)

### Define the Simplified Pipeline

**Original Pipeline**

`21 Features` → `Feature Engineering` → `Scaling` → `Feature Selection` → `9 Features` → `Model`

**Refitted Pipeline**

`9 Features` → `Feature Engineering` → `Scaling` → `Model`

- We've removed OneHotEncoder because its variable, BsmtFinType1, wasn't one of the selected features.
- We've also removed SelectFromModel because X_train and X_test now contain only those nine features identified during feature selection in the original pipeline.

In [ ]:
from sklearn.pipeline import Pipeline

# Feature Engineering
from feature_engine.encoding import OrdinalEncoder
from feature_engine import transformation as vt

# Feature Scaling
from sklearn.preprocessing import StandardScaler

# ML Algorithm
from sklearn.ensemble import ExtraTreesRegressor


def PipelineOptimization(model):
    """
    Creates the final regression pipeline using only the
    selected most important features.
    """

    pipeline_base = Pipeline([

        # CATEGORICAL ENCODING
        ("OrdinalCategoricalEncoder", OrdinalEncoder(
            encoding_method='arbitrary',
            variables=['KitchenQual'],
            unseen='encode'
        )),

        # NUMERICAL TRANSFORMATIONS
        ("LogTransformer", vt.LogTransformer(
            variables=[
                '1stFlrSF',
                'GrLivArea'
            ]
        )),

        ("PowerTransformer", vt.PowerTransformer(
            variables=['TotalBsmtSF']
        )),

        # FEATURE SCALING
        ("feat_scaling", StandardScaler()),

        # MODEL
        ("model", model),

    ])

    return pipeline_base

### Refit the Best Model

We now list the model that performed best, in this case, ExtraTreesClassifier.

In [ ]:
# Best performing model
models_search = {
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
}

# Best hyperparameters identified during the extensive search
params_search = {
    "ExtraTreesRegressor": {
        'model__max_depth': [15],
        'model__min_samples_leaf': [1],
        'model__min_samples_split': [50],
        'model__n_estimators': [100],
    }
}

models_search, params_search

and then refit

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_search,
    params=params_search
)

search.fit(
    X_train,
    y_train,
    scoring='r2',
    n_jobs=-1,
    cv=2
)

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(
    sort_by='mean_score'
)

grid_search_summary

In [ ]:
best_model = grid_search_summary.iloc[0, 0]
best_model

In [ ]:
best_pipeline = grid_search_pipelines[best_model].best_estimator_
best_pipeline

### Evaluate the Refitted Model

Finally, we evaluate the refitted pipeline to confirm that using only the selected features has not reduced model performance.

In [ ]:
regression_performance(
    X_train,
    y_train,
    X_test,
    y_test,
    best_pipeline
)

regression_evaluation_plots(
    X_train,
    y_train,
    X_test,
    y_test,
    best_pipeline,
    alpha_scatter=0.5
)

### Refit Results

The performance from this pipeline is the same as from the previous pipeline - as we should expect!

- R² score of **0.859** on the train set and **0.782** on the test set (i.e. the same as before)
- Therefore we are still meeting the client requirement of **0.75**

---

## 💾 10. Save Final Model and Outputs

The following will now be saved:
- Train set
- Test set
- Final regression pipeline
- Feature importance plot

In [ ]:
import joblib
import os

version = 'v1'
file_path = f'outputs/ml_pipeline/predict_house_prices/{version}'

try:
    os.makedirs(name=file_path)
except Exception as e:
    print(e)

### Train Set: features and target

Save the final train set.

In [ ]:
X_train.head()

In [ ]:
X_train.to_csv(f"{file_path}/X_train.csv", index=False)

In [ ]:
y_train

In [ ]:
y_train.to_csv(f"{file_path}/y_train.csv", index=False)

### Test Set: features and target

Save the final test set.

In [ ]:
X_test.head()

In [ ]:
X_test.to_csv(f"{file_path}/X_test.csv", index=False)

In [ ]:
y_test

In [ ]:
y_test.to_csv(f"{file_path}/y_test.csv", index=False)

### Modelling Pipeline

ML pipeline for predicting house sale prices.

In [ ]:
best_pipeline

In [ ]:
joblib.dump(
    value=best_pipeline,
    filename=f"{file_path}/reg_pipeline.pkl"
)

### Feature Importance Plot

Save the feature importance plot for use in the dashboard.

In [ ]:
df_feature_importance.plot(kind='bar', x='Feature', y='Importance')
plt.show()

In [ ]:
df_feature_importance.plot(kind='bar',x='Feature',y='Importance')
plt.savefig(f'{file_path}/features_importance.png', bbox_inches='tight')